In [ ]:
import os

# 대상 경로
base_dir = r"D:\sentinel-2\reducedImgData"

# 폴더 내 하위 폴더 목록 확인
folders = [name for name in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, name))]

# 출력
print("존재하는 폴더 목록:")
for folder in folders:
    print(folder)


In [ ]:
import os
import pandas as pd

# 경로 설정
base_dir = r"D:\sentinel-2\reducedImgData"
csv_file = os.path.join(base_dir, "in_matched_tif.csv")

# CSV에서 파일명 목록 읽기 (확장자 포함)
matched_tif_list = pd.read_csv(csv_file, header=None)[0].tolist()

# 결과 저장 리스트
matched_files = []

# base_dir 내 폴더들 순회
for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)
    dillingen_path = os.path.join(folder_path, "Dillingen")

    # Dillingen 폴더 존재 여부 확인
    if os.path.isdir(dillingen_path):
        # Dillingen 폴더 내 tif 파일 확인
        for file_name in os.listdir(dillingen_path):
            if file_name.lower().endswith('.tif') and file_name in matched_tif_list:
                full_path = os.path.join(dillingen_path, file_name)
                matched_files.append(full_path)

# 결과 출력
print("일치하는 tif 파일 목록:")
for f in matched_files:
    print(f)


In [ ]:
import os
import pandas as pd
import rasterio
from rasterio.enums import Resampling
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed

# 경로 설정
base_dir = r"D:\sentinel-2\reducedImgData"
csv_in = os.path.join(base_dir, "in_matched_tif.csv")
csv_op = os.path.join(base_dir, "op_matched_tif.csv")

# CSV 읽기
in_files_list = pd.read_csv(csv_in, header=None)[0].tolist()
op_files_list = pd.read_csv(csv_op, header=None)[0].tolist()

def normalize_to_255(array):
    array = np.nan_to_num(array, nan=0)
    min_val = np.percentile(array, 2)
    max_val = np.percentile(array, 98)
    norm_array = np.clip((array - min_val) / (max_val - min_val), 0, 1)
    return (norm_array * 255).astype(np.uint8)

def process_date(folder_path, dillingen_path, in_file, op_file):
    try:
        date_part = in_file.split('_')[2][:8]

        in_file_path = os.path.join(dillingen_path, in_file)
        op_file_path = os.path.join(dillingen_path, op_file)

        false_color_path = os.path.join(dillingen_path, "false color")
        os.makedirs(false_color_path, exist_ok=True)
        out_filename = f"{date_part}_12_8_4.tif"
        out_filepath = os.path.join(false_color_path, out_filename)
        out_png_path = os.path.join(false_color_path, f"{date_part}_12_8_4.png")

        with rasterio.open(in_file_path) as src_in:
            b1_in = src_in.read(1)
            b3_in = src_in.read(3)

            with rasterio.open(op_file_path) as src_op:
                b1_op = src_op.read(
                    1,
                    out_shape=(src_in.height, src_in.width),
                    resampling=Resampling.bilinear
                )

            out_meta = src_in.meta.copy()
            out_meta.update({"count": 3})

            stacked = np.stack([b1_in, b3_in, b1_op])

            with rasterio.open(out_filepath, 'w', **out_meta) as dst:
                dst.write(stacked)

            # ➡ PNG 저장
            rgb = np.stack([
                normalize_to_255(b1_in),
                normalize_to_255(b3_in),
                normalize_to_255(b1_op)
            ], axis=-1)

            plt.imsave(out_png_path, rgb)
            print(f"[성공] {out_filepath} & {out_png_path}")

    except Exception as e:
        print(f"[에러] {in_file} / {op_file} → {str(e)}")

# 작업 목록 생성
tasks = []

for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)
    dillingen_path = os.path.join(folder_path, "Dillingen")

    if not os.path.isdir(dillingen_path):
        continue

    tif_files = [f for f in os.listdir(dillingen_path) if f.lower().endswith('.tif')]

    for in_file in tif_files:
        if in_file not in in_files_list:
            continue

        try:
            date_part = in_file.split('_')[2][:8]
        except IndexError:
            print(f"[파일명 오류] {in_file}")
            continue

        op_file_match = None
        for op_file in tif_files:
            if op_file not in op_files_list:
                continue
            if date_part in op_file:
                op_file_match = op_file
                break

        if not op_file_match:
            print(f"[op 없음] {date_part}")
            continue

        tasks.append((folder_path, dillingen_path, in_file, op_file_match))

# 병렬 처리
print(f"\n✅ 총 처리 대상: {len(tasks)}개")
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_date, *task) for task in tasks]
    for future in as_completed(futures):
        _ = future.result()


In [ ]:
import os
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterio.plot import show

# KML 읽기
kml_file = r"E:\projects\smelter\shp\Blast Furnaces_v2.kml"
gdf = gpd.read_file(kml_file)

# 대상 base_dir
base_dir = r"D:\sentinel-2\reducedImgData"

# 각 Dillingen 폴더 순회
for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)
    dillingen_path = os.path.join(folder_path, "Dillingen", "false color")

    if not os.path.isdir(dillingen_path):
        continue

    # PNG 파일 순회
    for file in os.listdir(dillingen_path):
        if file.lower().endswith('.png'):
            png_path = os.path.join(dillingen_path, file)
            tif_name = file.replace('.png', '.tif')
            tif_path = os.path.join(dillingen_path, tif_name)

            if not os.path.exists(tif_path):
                print(f"[TIF 없음] {tif_path}")
                continue

            try:
                with rasterio.open(tif_path) as src:
                    fig, ax = plt.subplots(figsize=(12, 12))
                    show((src, [1, 2, 3]), ax=ax, adjust='linear')

                    # KML을 tif CRS로 변환
                    gdf_proj = gdf.to_crs(src.crs)

                    # ➡ 폴리곤 경계 기반 자동 확대
                    minx, miny, maxx, maxy = gdf_proj.total_bounds
                    ax.set_xlim(minx, maxx)
                    ax.set_ylim(miny, maxy)

                    # 폴리곤 테두리 흰색
                    gdf_proj.boundary.plot(ax=ax, edgecolor='white', linewidth=2)

                    ax.set_axis_off()

                    # 저장 (덮어쓰기)
                    plt.savefig(png_path, bbox_inches='tight', pad_inches=0, dpi=300)
                    plt.close()
                    print(f"[폴리곤 확대 완료] {png_path}")

            except Exception as e:
                print(f"[에러] {png_path} → {str(e)}")
